In [7]:
!pip install hdfs pyspark==3.5.1 --break-system-packages


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from hdfs import InsecureClient

# Nếu notebook ở host:
client = InsecureClient("http://namenode:9870", user="root")
client.status("/")

{'accessTime': 0,
 'blockSize': 0,
 'childrenNum': 8,
 'fileId': 16385,
 'group': 'supergroup',
 'length': 0,
 'modificationTime': 1773476530436,
 'owner': 'root',
 'pathSuffix': '',
 'permission': '755',
 'replication': 0,
 'snapshotEnabled': True,
 'storagePolicy': 0,
 'type': 'DIRECTORY'}

26/03/14 09:16:58 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
26/03/14 09:16:58 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/03/14 09:16:58 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
26/03/14 09:17:03 ERROR TaskSchedulerImpl: Lost executor 1 on 172.27.0.6: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
26/03/14 09:17:03 ERROR TaskSchedulerImpl: Lost executor 0 on 172.27.0.9: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
[Stage 0:>                                                          (0 + 0) / 1]

In [6]:

# Nếu notebook ở container:
# client = InsecureClient("http://namenode:9870", user="root")

# Upload file local -> HDFS
local_path = "./source_system/hadoop-3.4.0.tar.gz"
hdfs_path = "/tiktok/hadoop-3.4.0.tar.gz"

client.makedirs("/tiktok")
client.upload(hdfs_path, local_path, overwrite=True)
print(client.list("/tiktok"))


['hadoop-3.4.0.tar.gz']


In [7]:

# Nếu notebook ở container:
# client = InsecureClient("http://namenode:9870", user="root")

# Upload file local -> HDFS
local_path = "./source_system/run-wordcount.sh"
hdfs_path = "/tiktok/run-wordcount.sh"

client.makedirs("/tiktok")
client.upload(hdfs_path, local_path, overwrite=True)
print(client.list("/tiktok"))


['hadoop-3.4.0.tar.gz', 'run-wordcount.sh']


In [ ]:
# Download file to local then run.
client.download('/path/in/hdfs/file.txt', 'local_file.txt')

In [2]:
# Spark conf

In [9]:
from pyspark import SparkContext
from pyspark.sql import SparkSession

try:
    SparkSession.getActiveSession().stop()
except Exception:
    pass

try:
    sc = SparkContext.getOrCreate()
    sc.stop()
except Exception:
    pass
SparkContext._active_spark_context = None

spark = SparkSession.builder \
    .appName("kafka-to-hdfs") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

In [10]:
data = [
    (1, "Alice", 29, "Hanoi"),
    (2, "Bob", 35, "HCM"),
    (3, "Cindy", 22, "Danang"),
    (4, "David", 41, "Hanoi"),
]

df = spark.createDataFrame(data, ["id", "name", "age", "city"])
df.show()

26/03/14 09:20:27 WARN TaskSetManager: Lost task 0.0 in stage 0.0 (TID 0) (172.27.0.6 executor 1): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 8) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.

Py4JJavaError: An error occurred while calling o222.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 4 times, most recent failure: Lost task 0.3 in stage 0.0 (TID 3) (172.27.0.6 executor 1): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 8) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner

26/03/14 09:25:55 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
26/03/14 09:25:55 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/03/14 09:25:55 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
26/03/14 09:26:01 ERROR TaskSchedulerImpl: Lost executor 1 on 172.27.0.6: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
26/03/14 09:26:01 ERROR TaskSchedulerImpl: Lost executor 0 on 172.27.0.9: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
